# Day 1 Practice — Brute-Force RAG Chatbot

**Companion to:** `day1.ipynb` (Week 5 — RAG)

## What this notebook builds

A simple **Retrieval Augmented Generation (RAG)** assistant for Insurellm employees. It answers questions about company products and people by:

1. **Loading** markdown files from `knowledge-base/` into a Python dict
2. **Retrieving** relevant docs by matching words in the user's question to dict keys
3. **Injecting** those docs into the LLM system prompt
4. **Chatting** via a Gradio UI

## Architecture (high level)

```
User question
    → split into words → lookup in `knowledge` dict
    → append matches to system prompt
    → send to Claude (via OpenAI-compatible API)
    → return answer
```

## Key concepts to remember

| Term | Meaning here |
|------|--------------|
| **RAG** | Retrieve relevant docs, then generate an answer with the LLM |
| **Brute-force retrieval** | Exact word match against dict keys — no embeddings, no vector DB |
| **System prompt** | Instructions + injected context sent on every API call |
| **`knowledge` dict** | In-memory store: `{lookup_key: file_contents}` |

## Limitations of this approach (important for later weeks)

- Only finds docs when the user's **exact key word** appears (e.g. `lancaster`, `carllm`)
- Punctuation is stripped, but multi-word names like "Avery Lancaster" won't match unless you say `lancaster`
- No semantic search — "CEO" won't retrieve Avery Lancaster's record
- Context grows with every matched word (can get long / expensive)

## Prerequisites

- Run from the `week5/` directory (paths are relative)
- `.env` file with `ANTHROPIC_API_KEY` set (OpenAI key optional here)

In [1]:
# --- Imports ---
import os                          # read environment variables (API keys)
import glob                          # find files matching a pattern (e.g. knowledge-base/employees/*)
from dotenv import load_dotenv       # load .env file into os.environ
from openai import OpenAI            # OpenAI-compatible client (works with Anthropic via base_url)
from IPython.display import Markdown, display  # render LLM markdown responses in the notebook
from pathlib import Path             # clean path/filename handling (stem = filename without extension)
import gradio as gr                  # quick web UI for chat prototypes

## 1. Environment & API client setup

**Goal:** Load API keys and create an LLM client.

### `.env` and `load_dotenv(override=True)`
- Reads `OPENAI_API_KEY` and `ANTHROPIC_API_KEY` from a `.env` file in the project
- `override=True` means `.env` values **replace** any existing env vars (useful in notebooks where you re-run cells)

### Why use OpenAI client for Anthropic?
Anthropic exposes an **OpenAI-compatible** endpoint at `https://api.anthropic.com/v1/`.  
So we can reuse the same `OpenAI` class and `chat.completions.create()` pattern — just change `base_url` and `api_key`.

### Model choice
- `claude-haiku-4-5` — fast & cheap, good for prototyping
- Commented alternatives: `gpt-4.1-nano` (OpenAI), older Claude models

In [ ]:
# Setting up environment

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

# Sanity-check that keys are loaded (only prints first few chars — never log full keys)
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

# --- Model selection ---
# MODEL = "gpt-4.1-nano"              # OpenAI option (use openai = OpenAI() below)
MODEL = "claude-haiku-4-5"            # Anthropic — fast, low-cost
# MODEL = "claude-3-opus-20240229"    # more capable, slower/expensive
# MODEL = "claude-3-haiku-20240307"   # older Haiku

# --- LLM client ---
# OpenAI client is a thin wrapper around HTTP API calls.
# For Anthropic: point it at their OpenAI-compatible base URL.

# openai = OpenAI()                   # default OpenAI endpoint
anthropic_url = "https://api.anthropic.com/v1/"
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)


## 2. Quick API smoke test (optional)

**Goal:** Verify the Anthropic client works before building the full app.

### Chat completions message format
```python
messages = [{"role": "user", "content": "..."}]
response = anthropic.chat.completions.create(model=MODEL, messages=messages)
answer = response.choices[0].message.content
```

- `role` can be `"system"`, `"user"`, or `"assistant"`
- `response.choices[0]` — first (usually only) completion
- `display(Markdown(...))` renders the answer nicely in Jupyter

Uncomment the cell below to run the test.

In [ ]:
# Smoke test — uncomment to verify API connection works
# tell_a_joke = [
#     {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
# ]
# response = anthropic.chat.completions.create(model=MODEL, messages=tell_a_joke)
# display(Markdown(response.choices[0].message.content))

## 3. List available models (optional)

**Goal:** Discover valid model IDs via Anthropic's native SDK.

The OpenAI-compatible client doesn't expose a model list, so we use the official `anthropic` package here.  
`native_client.models.list()` returns all models your API key can access.

Uncomment to print model IDs (useful when a model name returns 404).

In [ ]:
# List Anthropic models — uncomment if you need to check valid model IDs
# import anthropic as anthropic_sdk
#
# native_client = anthropic_sdk.Anthropic()  # reads ANTHROPIC_API_KEY from env
# for m in native_client.models.list():
#     print(m.id)

## 4. Build the knowledge base — employees

**Goal:** Load all employee HR markdown files into a single in-memory dictionary.

### How it works
| Step | What happens |
|------|--------------|
| `glob.glob("knowledge-base/employees/*")` | Returns all file paths in that folder |
| `Path(filename).stem` | Filename without extension, e.g. `"Avery Lancaster"` |
| `.split(' ')[-1]` | Takes **last word** as the lookup key → `"Lancaster"` |
| `.lower()` | Normalizes key → `"lancaster"` |
| `f.read()` | Entire file content becomes the dict **value** |

### Example
- File: `knowledge-base/employees/Avery Lancaster.md`
- Key: `"lancaster"`
- Value: full markdown HR record as a string

### Why last name as key?
Users will likely ask *"Who is Lancaster?"* — the retrieval step matches individual words, so `lancaster` must be a dict key.

In [ ]:
knowledge = {}  # our in-memory "database": {lookup_key: document_text}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    # "Avery Lancaster.md" → stem "Avery Lancaster" → last word "Lancaster" → key "lancaster"
    last_name = Path(filename).stem.split(' ')[-1]
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[last_name.lower()] = f.read()


## 5. Build the knowledge base — products

**Goal:** Add product docs to the **same** `knowledge` dict.

### Key difference from employees
- Product filenames are single words: `Carllm.md`, `Homellm.md`
- We use the **full stem** (filename without `.md`) as the key
- `.lower()` → `"carllm"`, `"homellm"`, etc.

After this cell, `knowledge` holds both employees and products.  
You can inspect keys with `knowledge.keys()` or test a lookup: `knowledge["lancaster"]`.

In [ ]:
filenames = glob.glob("knowledge-base/products/*")

for filename in filenames:
    # "Carllm.md" → stem "Carllm" → key "carllm"
    name = Path(filename).stem
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

## 6. Retrieval — `get_relevant_context()` (the "R" in RAG)

**Goal:** Given a user message, find matching documents from `knowledge`.

### Step-by-step
1. **Strip non-letters** — `''.join(ch for ch in message if ch.isalpha() or ch.isspace())`  
   Removes `?`, `,`, etc. so `"carllm?"` → `"carllm"`
2. **Tokenize** — `.lower().split()` → list of words
3. **Lookup** — for each word, if it's a key in `knowledge`, append its value

### List comprehension (pythonic version)
```python
[knowledge[word] for word in words if word in knowledge]
```
Equivalent to a for-loop that only collects hits — skips words with no match.

### Test query below
`"Who is Lancaster and what is carllm?"` → words include `lancaster` and `carllm` → returns **two** document strings.

### What this does NOT do
- No fuzzy matching, synonyms, or embeddings
- `"CEO"` won't find Lancaster; `"Avery"` won't either (key is `lancaster`)
- Common words like `"who"`, `"what"` are harmlessly ignored (not in dict)

In [ ]:
def get_relevant_context(message):
    """Return a list of document strings whose keys appear as words in message."""
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[word] for word in words if word in knowledge]

# Demo: should return [lancaster_hr_record, carllm_product_doc]
get_relevant_context("Who is Lancaster and what is carllm?")

## 7. Format context for the prompt — `additional_context()`

**Goal:** Turn the list of retrieved docs into a single string for the system prompt.

### Two outcomes
| Situation | Result |
|-----------|--------|
| No matches | `"There is no additional context relevant..."` — LLM answers from general knowledge only |
| Matches found | Header + docs joined with `\n\n` (double newline separates documents) |

### Why a wrapper function?
Keeps `chat()` clean. The system prompt always gets a consistent text block whether or not retrieval found anything.

Try: `print(additional_context("Who is Alex Lancaster?"))` — note `"alex"` won't match; only `"lancaster"` would.

In [ ]:
def additional_context(message):
    """Format retrieved docs as a string to append to the system prompt."""
    relevant_context = get_relevant_context(message)
    if not relevant_context:
        result = "There is no additional context relevant to the user's question."
    else:
        result = "The following additional context might be relevant in answering the user's question:\n\n"
        result += "\n\n".join(relevant_context)  # separate multiple docs with blank lines
    return result

## 8. System prompt — `SYSTEM_PREFIX`

**Goal:** Define the assistant's persona and reserve space for injected RAG context.

### Structure
```
[Fixed instructions about Insurellm]
Relevant context:
[Dynamic content from additional_context() appended at runtime]
```

### Key instructions to the model
- Represent Insurellm (insurance tech company)
- Answer about employees and products
- Use the provided context
- Be brief; admit when unsure

At call time: `system_message = SYSTEM_PREFIX + additional_context(message)`

The model sees retrieved HR/product docs **inside** the system message — not as a separate user message.

In [ ]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an exper in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""
# Note: additional_context(message) is concatenated to the end of this string in chat()

## 9. Chat function — wiring RAG + LLM

**Goal:** The function Gradio calls on every user message.

### Message array sent to the API
```python
[
  {"role": "system", "content": SYSTEM_PREFIX + retrieved_docs},
  ...history...,                              # prior user/assistant turns
  {"role": "user", "content": current_message}
]
```

### Parameters
| Param | Purpose |
|-------|---------|
| `message` | Current user input (also used for retrieval) |
| `history` | Previous chat turns — Gradio passes this automatically with `type="messages"` |

### Flow per message
1. Retrieve context from `message`
2. Build full system prompt
3. Prepend system + append history + current user message
4. Call `anthropic.chat.completions.create()`
5. Return assistant text string

In [ ]:
def chat(message, history):
    """Gradio callback: retrieve context, call LLM, return assistant reply."""
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = (
        [{"role": "system", "content": system_message}]
        + history
        + [{"role": "user", "content": message}]
    )
    response = anthropic.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

## 10. Gradio Chat UI

**Goal:** Launch a browser chat interface in one line.

```python
gr.ChatInterface(chat, type="messages").launch(inbrowser=True)
```

### What this does
- `ChatInterface` — pre-built chat UI; calls your `chat(message, history)` function
- `type="messages"` — history format is OpenAI-style dicts (`{"role": ..., "content": ...}`)
- `launch(inbrowser=True)` — starts local server and opens a browser tab

### Example questions to try
- `"Who is Lancaster?"` — retrieves employee record
- `"What is carllm?"` — retrieves product doc
- `"Who is Lancaster and what is carllm?"` — retrieves both
- `"Who is the CEO?"` — likely **no** retrieval (no key match) — model may hallucinate

### To stop the server
Interrupt the Jupyter kernel (Stop button) when done.

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

---

## Quick reference / lookup cheat sheet

### File → dict key mapping
| Source folder | Filename example | Dict key |
|---------------|------------------|----------|
| `knowledge-base/employees/` | `Avery Lancaster.md` | `lancaster` |
| `knowledge-base/employees/` | `Michael O'Brien.md` | `o'brien` |
| `knowledge-base/products/` | `Carllm.md` | `carllm` |

### RAG pipeline (one line each)
1. **Load:** `glob` → read files → `knowledge[key] = content`
2. **Retrieve:** split user message into words → dict lookup
3. **Augment:** append hits to `SYSTEM_PREFIX`
4. **Generate:** `chat.completions.create(messages=...)`

### Useful debug one-liners
```python
knowledge.keys()                                    # all lookup keys
knowledge["lancaster"][:200]                        # peek at a document
get_relevant_context("your question here")          # see what gets retrieved
print(additional_context("your question here"))     # see full injected context
```

### Common gotchas
- **Run from `week5/`** — relative paths won't work from repo root
- **Exact word match only** — ask using keys (`lancaster`, `carllm`), not roles (`CEO`)
- **Re-run knowledge cells** after adding new files to `knowledge-base/`
- **Gradio blocks the cell** — kernel stays busy until you stop it
- **Token cost** — large retrieved docs in every system message add up fast

### What's next in the course
Later days replace brute-force lookup with **embeddings + vector search** (Chroma, etc.) for semantic retrieval.